# Part B, version 2: drought index and rainfall-vegetation link in the Sokoto-Rima basin

This notebook replaces the parts of `sokoto_rima_drought.ipynb` that a review in November 2026 found
wrong. The original notebook is kept unchanged because the October 2026 report refers to it.

What changes:
1. **SPI is gamma-fitted** (McKee et al., 1993): a gamma distribution is fitted to annual rainfall totals
   and each year's probability is converted to a standard normal value. The original "SPI" was a z-score.
2. **The NDVI-rainfall correlation is reported raw, detrended and on year-to-year differences**, because
   both series rise with the Sahel recovery and a shared trend inflates a raw correlation.
3. **NDVI uses quality-screened MOD13Q1** (250 m, SummaryQA 0-1), averaged over the growing season
   (July-October) of complete years only. The original averaged every composite of the year without
   quality screening, and included the partial year 2000.
4. Years are calendar years, and the unused machine-learning imports are gone.

Run every cell, then download `part_b_outputs.zip` (last cell) and commit its files to `part_b/outputs/`.

In [ ]:
# Set your own Earth Engine Cloud project here
GEE_PROJECT = "YOUR-EARTH-ENGINE-PROJECT"

import os, json
import ee
import numpy as np
import pandas as pd
from scipy import stats

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    # Sign-in by link and code: works even when the browser blocks Colab's pop-up sign-in
    # ("credential propagation was unsuccessful"). Open the link, choose your account,
    # copy the code it shows and paste it into the box below this cell.
    ee.Authenticate(auth_mode="notebook", force=True)
    ee.Initialize(project=GEE_PROJECT)

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
print("Earth Engine ready; results will be written to", os.path.abspath(OUT))

In [ ]:
basin = ee.Feature(ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_4")
                   .filterBounds(ee.Geometry.Point([5.23, 13.06])).first())
print("HydroBASINS unit:", basin.get("HYBAS_ID").getInfo())   # expected 1040760290
GEOM = basin.geometry()
YEARS = list(range(2001, 2025))   # complete years of MODIS

## 1. Annual rainfall and gamma-fitted SPI (CHIRPS daily)

In [ ]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
rain = []
for y in range(1981, 2025):
    tot = chirps.filterDate(f"{y}-01-01", f"{y+1}-01-01").sum().reduceRegion(
        ee.Reducer.mean(), GEOM, 5566, maxPixels=1e10).get("precipitation").getInfo()
    rain.append(dict(year=y, rain_mm=tot))
rain = pd.DataFrame(rain)

def spi_gamma(x, ref):
    """SPI by McKee et al. (1993): gamma fitted to the reference years, then mapped to a standard normal."""
    a, loc, scale = stats.gamma.fit(ref, floc=0)
    return stats.norm.ppf(stats.gamma.cdf(x, a, loc=0, scale=scale)), (a, scale)

rain["spi"], (shape_, scale_) = spi_gamma(rain["rain_mm"], rain.loc[rain.year.between(1991, 2020), "rain_mm"])
rain["z_score_old_method"] = (rain["rain_mm"] - rain["rain_mm"].mean()) / rain["rain_mm"].std()
rain.to_csv(f"{OUT}/rain_spi_1981_2024.csv", index=False)
print(f"gamma shape {shape_:.1f}, scale {scale_:.2f}; reference 1991-2020")
rain.tail()

## 2. Growing-season NDVI (MOD13Q1, quality-screened)

In [ ]:
def ndvi(img):
    return img.select("NDVI").updateMask(img.select("SummaryQA").lte(1)).multiply(0.0001).copyProperties(img, ["system:time_start"])

mod = ee.ImageCollection("MODIS/061/MOD13Q1").map(ndvi)
veg = []
for y in YEARS:
    gs = mod.filterDate(f"{y}-07-01", f"{y}-11-01").mean().reduceRegion(ee.Reducer.mean(), GEOM, 1000, maxPixels=1e10).get("NDVI").getInfo()
    yr = mod.filterDate(f"{y}-01-01", f"{y+1}-01-01").mean().reduceRegion(ee.Reducer.mean(), GEOM, 1000, maxPixels=1e10).get("NDVI").getInfo()
    veg.append(dict(year=y, ndvi_growing_season=gs, ndvi_annual_mean=yr))
veg = pd.DataFrame(veg)
veg.to_csv(f"{OUT}/ndvi_2001_2024.csv", index=False)
veg.tail()

## 3. Rainfall-NDVI correlation: raw, detrended and differenced

Detrending removes a straight-line trend from each series; differencing compares each year with the one
before. If the raw correlation is much stronger than the other two, most of it comes from the shared
trend rather than from year-to-year response of vegetation to rain.

In [ ]:
from scipy.signal import detrend
m = rain.merge(veg, on="year")
x, y = m["rain_mm"].to_numpy(), m["ndvi_growing_season"].to_numpy()
def corr(a, b):
    r, p = stats.pearsonr(a, b); rho, ps = stats.spearmanr(a, b)
    return {"pearson_r": r, "pearson_p": p, "spearman_rho": rho, "spearman_p": ps, "n": len(a)}
result = {"years": [int(m.year.min()), int(m.year.max())],
          "raw": corr(x, y), "detrended": corr(detrend(x), detrend(y)), "differenced": corr(np.diff(x), np.diff(y)),
          "trend_rain_mm_per_year": stats.linregress(m.year, x).slope,
          "trend_ndvi_per_decade": 10 * stats.linregress(m.year, y).slope,
          "spi_reference": "1991-2020", "ndvi": "MOD13Q1 SummaryQA<=1, July-October mean"}
json.dump(result, open(f"{OUT}/part_b_results.json", "w"), indent=1, default=float)
pd.DataFrame({k: result[k] for k in ("raw", "detrended", "differenced")}).T

In [ ]:
# In Colab: download the outputs folder, then commit its files to part_b/outputs/ in the repository.
import shutil
shutil.make_archive("part_b_outputs", "zip", OUT)
try:
    from google.colab import files
    files.download("part_b_outputs.zip")
except ImportError:
    print("Not in Colab: the results are in", os.path.abspath(OUT))